In [1]:
import pandas as pd

df = pd.read_csv('research/phase0/data/MERGED_XAUUSDm_M1_2020_01_01_to_2026_05_07.csv')
df['Time'] = pd.to_datetime(df['Time'], format='%Y.%m.%d %H:%M')
df = df.set_index('Time').sort_index()

print(f"Total rows: {len(df):,}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"Duplicates: {df.index.duplicated().sum()}")

# Find gaps larger than 4 hours (filters out normal 1-min gaps)
time_diffs = df.index.to_series().diff()
gaps = time_diffs[time_diffs > pd.Timedelta(hours=4)]

# Categorize gaps
weekend_gaps = gaps[(gaps >= pd.Timedelta(hours=40)) & (gaps <= pd.Timedelta(hours=56))]
suspicious_gaps = gaps[(gaps > pd.Timedelta(hours=4)) & (gaps < pd.Timedelta(hours=40))]
huge_gaps = gaps[gaps > pd.Timedelta(hours=56)]

print(f"\nWeekend gaps (40-56 hours, normal):  {len(weekend_gaps)}")
print(f"Suspicious gaps (4-40 hours):        {len(suspicious_gaps)}")
print(f"Huge gaps (>56 hours, missing data): {len(huge_gaps)}")

# Show only the problematic gaps
if len(suspicious_gaps) > 0:
    print("\n=== SUSPICIOUS GAPS (4-40 hours) ===")
    print("These might indicate missing trading hours or merge issues:")
    for ts, gap in suspicious_gaps.items():
        hours = gap.total_seconds() / 3600
        print(f"  {ts}  ->  gap of {hours:.1f} hours")

if len(huge_gaps) > 0:
    print("\n=== HUGE GAPS (>2.5 days) ===")
    print("These indicate missing data chunks or extended market closures:")
    for ts, gap in huge_gaps.items():
        days = gap.total_seconds() / 86400
        print(f"  {ts}  ->  gap of {days:.1f} days")

# Rows per year — quick visual check
print("\n=== ROWS PER YEAR ===")
yearly = df.groupby(df.index.year).size()
for year, count in yearly.items():
    bar = '#' * int(count / 10000)
    print(f"  {year}: {count:>7,}  {bar}")

# Rows per month for last 24 months
print("\n=== ROWS PER MONTH (last 24 months) ===")
monthly = df.groupby([df.index.year, df.index.month]).size().tail(24)
for (year, month), count in monthly.items():
    bar = '#' * int(count / 1000)
    print(f"  {year}-{month:02d}: {count:>6,}  {bar}")

Total rows: 2,240,449
Date range: 2020-01-02 01:00:00 to 2026-05-07 14:16:00
Duplicates: 0

Weekend gaps (40-56 hours, normal):  316
Suspicious gaps (4-40 hours):        28
Huge gaps (>56 hours, missing data): 15

=== SUSPICIOUS GAPS (4-40 hours) ===
These might indicate missing trading hours or merge issues:
  2020-01-20 23:05:00  ->  gap of 5.1 hours
  2020-02-17 23:01:00  ->  gap of 5.0 hours
  2020-05-25 22:05:00  ->  gap of 5.1 hours
  2020-09-07 22:01:00  ->  gap of 5.0 hours
  2020-11-26 23:01:00  ->  gap of 5.0 hours
  2021-01-18 23:01:00  ->  gap of 5.3 hours
  2021-02-15 23:01:00  ->  gap of 5.3 hours
  2021-05-31 22:01:00  ->  gap of 5.0 hours
  2021-07-05 22:05:00  ->  gap of 5.1 hours
  2021-09-06 22:01:00  ->  gap of 5.0 hours
  2021-11-25 23:01:00  ->  gap of 5.3 hours
  2022-05-30 22:01:00  ->  gap of 4.8 hours
  2022-06-20 22:01:00  ->  gap of 4.8 hours
  2022-07-04 22:01:00  ->  gap of 4.8 hours
  2022-09-05 22:01:00  ->  gap of 4.8 hours
  2023-05-29 22:01:00  ->  ga

In [2]:
# === DATA SPLIT FOR HONEST RESEARCH ===

# Define cutoff dates
TRAINING_END = '2024-12-31 23:59:59'
VALIDATION_END = '2025-09-30 23:59:59'

# Split the data
training = df.loc[:TRAINING_END].copy()
validation = df.loc[TRAINING_END:VALIDATION_END].copy()
holdout = df.loc[VALIDATION_END:].copy()

print("=== DATA SPLIT ===\n")
print(f"TRAINING SET (we explore freely on this):")
print(f"  Period: {training.index.min()} to {training.index.max()}")
print(f"  Rows:   {len(training):,}")
print(f"  Years:  {(training.index.max() - training.index.min()).days / 365:.1f}")

print(f"\nVALIDATION SET (used to compare final candidates):")
print(f"  Period: {validation.index.min()} to {validation.index.max()}")
print(f"  Rows:   {len(validation):,}")
print(f"  Months: {(validation.index.max() - validation.index.min()).days / 30:.1f}")

print(f"\nHOLDOUT SET (NEVER analyzed until end):")
print(f"  Period: {holdout.index.min()} to {holdout.index.max()}")
print(f"  Rows:   {len(holdout):,}")
print(f"  Months: {(holdout.index.max() - holdout.index.min()).days / 30:.1f}")

# Save holdout to a separate file so it's physically isolated
holdout.to_csv('research/phase0/data/HOLDOUT_DO_NOT_TOUCH_UNTIL_FINAL.csv')
print(f"\nHoldout saved to disk and removed from active memory.")

# Also save validation separately for clarity
validation.to_csv('research/phase0/data/VALIDATION_USE_SPARINGLY.csv')
print(f"Validation saved to disk for later use.")

# Free memory — we work only with training going forward
del holdout
del validation
del df  # remove the full set so we can't accidentally use it

# Rename training to df so all our existing code works without changes
df = training
del training

print(f"\nActive working dataset: {len(df):,} rows")
print(f"Period: {df.index.min()} to {df.index.max()}")
print(f"\nReady for analysis.")

=== DATA SPLIT ===

TRAINING SET (we explore freely on this):
  Period: 2020-01-02 01:00:00 to 2024-12-31 21:57:00
  Rows:   1,767,439
  Years:  5.0

VALIDATION SET (used to compare final candidates):
  Period: 2025-01-01 23:05:00 to 2025-09-30 23:59:00
  Rows:   262,474
  Months: 9.1

HOLDOUT SET (NEVER analyzed until end):
  Period: 2025-10-01 00:00:00 to 2026-05-07 14:16:00
  Rows:   210,536
  Months: 7.3

Holdout saved to disk and removed from active memory.
Validation saved to disk for later use.

Active working dataset: 1,767,439 rows
Period: 2020-01-02 01:00:00 to 2024-12-31 21:57:00

Ready for analysis.


In [3]:
# === MARKET REGIME OVERVIEW (training set only) ===

yearly = df.groupby(df.index.year).agg(
    open=('Open', 'first'),
    close=('Close', 'last'),
    low=('Low', 'min'),
    high=('High', 'max'),
)
yearly['return_pct'] = ((yearly['close'] / yearly['open']) - 1) * 100
yearly['range_pct'] = ((yearly['high'] - yearly['low']) / yearly['open']) * 100

print("=== YEARLY GOLD REGIMES IN TRAINING DATA ===\n")
print(yearly.to_string())

print("\n\n=== INTERPRETATION ===")
for year, row in yearly.iterrows():
    ret = row['return_pct']
    if ret > 15:
        regime = "STRONG BULL"
    elif ret > 5:
        regime = "BULL"
    elif ret > -5:
        regime = "RANGING"
    elif ret > -15:
        regime = "BEAR"
    else:
        regime = "STRONG BEAR"
    print(f"  {year}: {ret:+6.1f}%  →  {regime}")

=== YEARLY GOLD REGIMES IN TRAINING DATA ===

          open     close       low      high  return_pct  range_pct
Time                                                               
2020  1518.464  1894.387  1451.281  2073.682   24.756794  40.988855
2021  1909.308  1827.615  1676.627  1959.290   -4.278671  14.804474
2022  1830.615  1823.510  1614.427  2070.292   -0.388121  24.902287
2023  1826.216  2062.838  1804.393  2144.501   12.956956  18.623646
2024  2064.593  2624.381  1984.121  2790.091   27.113722  39.037718


=== INTERPRETATION ===
  2020:  +24.8%  →  STRONG BULL
  2021:   -4.3%  →  RANGING
  2022:   -0.4%  →  RANGING
  2023:  +13.0%  →  BULL
  2024:  +27.1%  →  STRONG BULL


In [4]:
yearly = df.groupby(df.index.year).agg(
    open=('Open', 'first'),
    close=('Close', 'last'),
    low=('Low', 'min'),
    high=('High', 'max'),
)
yearly['return_pct'] = ((yearly['close'] / yearly['open']) - 1) * 100
yearly['range_pct'] = ((yearly['high'] - yearly['low']) / yearly['open']) * 100

print("=== YEARLY GOLD REGIMES IN TRAINING DATA ===\n")
print(yearly.to_string())

print("\n\n=== INTERPRETATION ===")
for year, row in yearly.iterrows():
    ret = row['return_pct']
    if ret > 15:
        regime = "STRONG BULL"
    elif ret > 5:
        regime = "BULL"
    elif ret > -5:
        regime = "RANGING"
    elif ret > -15:
        regime = "BEAR"
    else:
        regime = "STRONG BEAR"
    print(f"  {year}: {ret:+6.1f}%  →  {regime}")

=== YEARLY GOLD REGIMES IN TRAINING DATA ===

          open     close       low      high  return_pct  range_pct
Time                                                               
2020  1518.464  1894.387  1451.281  2073.682   24.756794  40.988855
2021  1909.308  1827.615  1676.627  1959.290   -4.278671  14.804474
2022  1830.615  1823.510  1614.427  2070.292   -0.388121  24.902287
2023  1826.216  2062.838  1804.393  2144.501   12.956956  18.623646
2024  2064.593  2624.381  1984.121  2790.091   27.113722  39.037718


=== INTERPRETATION ===
  2020:  +24.8%  →  STRONG BULL
  2021:   -4.3%  →  RANGING
  2022:   -0.4%  →  RANGING
  2023:  +13.0%  →  BULL
  2024:  +27.1%  →  STRONG BULL


In [5]:
# === H4 ON FULL TRAINING DATA — BOTH DIRECTIONS, BY YEAR ===

# First, recalculate indicators on the full training set
def calculate_atr(data, period=14):
    high_low = data['High'] - data['Low']
    high_close = abs(data['High'] - data['Close'].shift(1))
    low_close = abs(data['Low'] - data['Close'].shift(1))
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()

def calculate_session_vwap(data, session_start_hour=13):
    data = data.copy()
    data['typical'] = (data['High'] + data['Low'] + data['Close']) / 3
    data['tpv'] = data['typical'] * data['TickVolume']
    data['session_id'] = (
        (data.index.hour == session_start_hour) & 
        (data.index.minute == 0)
    ).cumsum()
    data['cum_tpv'] = data.groupby('session_id')['tpv'].cumsum()
    data['cum_vol'] = data.groupby('session_id')['TickVolume'].cumsum()
    return data['cum_tpv'] / data['cum_vol']

print("Computing indicators on 1.77M rows... (will take 30-60 seconds)")
df['spread_usd'] = df['Spread'] * 0.001
df['atr14'] = calculate_atr(df, 14)
df['vwap'] = calculate_session_vwap(df, session_start_hour=13)
df['dist_from_vwap'] = df['Close'] - df['vwap']
df['hour'] = df.index.hour
df['bullish_candle'] = df['Close'] > df['Open']
df['bearish_candle'] = df['Close'] < df['Open']
print("Done.\n")

# H4 parameters (from this afternoon)
ATR_MULT_FROM_VWAP = 2.5
ATR_MULT_SL        = 1.0
MAX_SPREAD         = 0.40   # slightly wider since some periods may have higher spreads
TIMEOUT_MIN        = 25
TRADE_HOURS        = [13, 14]
ENTRY_COST_USD     = 0.33
MIN_TP_ATR         = 1.0
TP_VWAP_FRACTION   = 0.7

trades_h4_full = []
i = 50
total = len(df)

while i < total - TIMEOUT_MIN:
    row = df.iloc[i]
    
    if row['hour'] not in TRADE_HOURS:
        i += 1
        continue
    if row['spread_usd'] > MAX_SPREAD:
        i += 1
        continue
    if pd.isna(row['atr14']) or row['atr14'] < 0.4:
        i += 1
        continue
    if pd.isna(row['vwap']):
        i += 1
        continue
    
    atr = row['atr14']
    dist = row['dist_from_vwap']
    
    direction = None
    if dist > ATR_MULT_FROM_VWAP * atr and row['bearish_candle']:
        direction = 'SELL'
    elif dist < -ATR_MULT_FROM_VWAP * atr and row['bullish_candle']:
        direction = 'BUY'
    
    if direction is None:
        i += 1
        continue
    
    entry_price = row['Close']
    entry_time = df.index[i]
    target_distance = max(abs(dist) * TP_VWAP_FRACTION, MIN_TP_ATR * atr)
    
    if direction == 'BUY':
        sl = entry_price - ATR_MULT_SL * atr
        tp = entry_price + target_distance
    else:
        sl = entry_price + ATR_MULT_SL * atr
        tp = entry_price - target_distance
    
    exit_price, exit_reason, exit_time = None, None, None
    
    for j in range(i + 1, min(i + 1 + TIMEOUT_MIN, total)):
        bar = df.iloc[j]
        if direction == 'BUY':
            if bar['Low'] <= sl:
                exit_price, exit_reason, exit_time = sl, 'SL', df.index[j]
                break
            if bar['High'] >= tp:
                exit_price, exit_reason, exit_time = tp, 'TP', df.index[j]
                break
        else:
            if bar['High'] >= sl:
                exit_price, exit_reason, exit_time = sl, 'SL', df.index[j]
                break
            if bar['Low'] <= tp:
                exit_price, exit_reason, exit_time = tp, 'TP', df.index[j]
                break
    
    if exit_price is None:
        exit_idx = min(i + TIMEOUT_MIN, total - 1)
        exit_price = df.iloc[exit_idx]['Close']
        exit_reason = 'TIMEOUT'
        exit_time = df.index[exit_idx]
    
    if direction == 'BUY':
        gross_pl = exit_price - entry_price
    else:
        gross_pl = entry_price - exit_price
    
    net_pl = gross_pl - (ENTRY_COST_USD * 2)
    
    trades_h4_full.append({
        'entry_time': entry_time,
        'year': entry_time.year,
        'direction': direction,
        'entry': entry_price,
        'sl': sl,
        'tp': tp,
        'exit_time': exit_time,
        'exit': exit_price,
        'exit_reason': exit_reason,
        'atr': atr,
        'dist_from_vwap': dist,
        'gross_pl_usd': gross_pl,
        'net_pl_usd': net_pl,
        'hour': row['hour']
    })
    
    i = j + 1 if exit_price is not None else i + 1

trades_h4_full_df = pd.DataFrame(trades_h4_full)

print(f"=== H4 ON 5 YEARS OF TRAINING DATA ===\n")
print(f"Total trades: {len(trades_h4_full_df):,}\n")

# Overall stats
def summarize(name, t):
    if len(t) == 0:
        print(f"\n{name}: NO TRADES")
        return None
    wins = t[t['net_pl_usd'] > 0]
    losses = t[t['net_pl_usd'] <= 0]
    pf = wins['net_pl_usd'].sum() / abs(losses['net_pl_usd'].sum()) if len(losses) > 0 and losses['net_pl_usd'].sum() != 0 else float('inf')
    print(f"{name}:")
    print(f"  Trades:     {len(t):,}")
    print(f"  Win rate:   {(t['net_pl_usd']>0).mean()*100:.1f}%")
    print(f"  PF:         {pf:.2f}")
    print(f"  Total P/L:  ${t['net_pl_usd'].sum():.2f}")
    print(f"  Per trade:  ${t['net_pl_usd'].mean():.2f}")
    return pf

print("=== OVERALL ===")
summarize("All trades", trades_h4_full_df)
print()
summarize("BUY only", trades_h4_full_df[trades_h4_full_df['direction'] == 'BUY'])
print()
summarize("SELL only", trades_h4_full_df[trades_h4_full_df['direction'] == 'SELL'])

# By year × direction (the most important table)
print("\n\n=== BY YEAR × DIRECTION ===")
print(f"{'Year':<8}{'Direction':<12}{'Trades':<10}{'Win%':<10}{'PF':<8}{'Total P/L':<12}{'PerTrade':<10}")
print("-" * 70)
for year in sorted(trades_h4_full_df['year'].unique()):
    for direction in ['BUY', 'SELL']:
        subset = trades_h4_full_df[
            (trades_h4_full_df['year'] == year) & 
            (trades_h4_full_df['direction'] == direction)
        ]
        if len(subset) == 0:
            continue
        wins = subset[subset['net_pl_usd'] > 0]
        losses = subset[subset['net_pl_usd'] <= 0]
        pf = wins['net_pl_usd'].sum() / abs(losses['net_pl_usd'].sum()) if len(losses) > 0 and losses['net_pl_usd'].sum() != 0 else float('inf')
        wr = (subset['net_pl_usd'] > 0).mean() * 100
        total = subset['net_pl_usd'].sum()
        per = subset['net_pl_usd'].mean()
        print(f"{year:<8}{direction:<12}{len(subset):<10}{wr:<10.1f}{pf:<8.2f}{total:<12.2f}{per:<10.2f}")

Computing indicators on 1.77M rows... (will take 30-60 seconds)
Done.

=== H4 ON 5 YEARS OF TRAINING DATA ===

Total trades: 6,604

=== OVERALL ===
All trades:
  Trades:     6,604
  Win rate:   28.1%
  PF:         0.47
  Total P/L:  $-4543.56
  Per trade:  $-0.69

BUY only:
  Trades:     3,085
  Win rate:   29.2%
  PF:         0.50
  Total P/L:  $-2044.46
  Per trade:  $-0.66

SELL only:
  Trades:     3,519
  Win rate:   27.2%
  PF:         0.44
  Total P/L:  $-2499.10
  Per trade:  $-0.71


=== BY YEAR × DIRECTION ===
Year    Direction   Trades    Win%      PF      Total P/L   PerTrade  
----------------------------------------------------------------------
2020    BUY         500       27.8      0.52    -356.75     -0.71     
2020    SELL        716       26.7      0.41    -538.19     -0.75     
2021    BUY         618       30.9      0.53    -357.78     -0.58     
2021    SELL        687       29.1      0.44    -448.63     -0.65     
2022    BUY         655       29.6      0.49    -

In [6]:
# === RESAMPLE M1 TO M5 (TRAINING DATA ONLY) ===

# Aggregation rules for OHLC + volume + spread
m5 = df.resample('5min').agg({
    'Open':       'first',
    'High':       'max',
    'Low':        'min',
    'Close':      'last',
    'TickVolume': 'sum',
    'Spread':     'mean',  # average spread across the 5 minutes
    'RealVolume': 'sum'
})

# Drop bars with no data (weekends, holidays produce all-NaN rows)
m5 = m5.dropna(subset=['Open', 'High', 'Low', 'Close'])

# Recompute USD spread
m5['spread_usd'] = m5['Spread'] * 0.001

print(f"=== M5 TRAINING DATA ===\n")
print(f"M1 rows: {len(df):,}")
print(f"M5 rows: {len(m5):,}")
print(f"Period: {m5.index.min()} to {m5.index.max()}")

# Compare M1 vs M5 candle stats
print(f"\n=== CANDLE RANGE COMPARISON (USD) ===")
m5['range_usd'] = m5['High'] - m5['Low']
m1_range = (df['High'] - df['Low'])
print(f"\nM1 candle range:")
print(m1_range.describe()[['mean', '50%', '75%', '95%']].round(3).to_string())
print(f"\nM5 candle range:")
print(m5['range_usd'].describe()[['mean', '50%', '75%', '95%']].round(3).to_string())

print(f"\n=== COST-TO-MOVEMENT RATIO ===")
m1_median = m1_range.median()
m5_median = m5['range_usd'].median()
round_trip_cost = 0.66
print(f"M1: median range ${m1_median:.2f}, cost ${round_trip_cost:.2f} = cost is {round_trip_cost/m1_median*100:.1f}% of range")
print(f"M5: median range ${m5_median:.2f}, cost ${round_trip_cost:.2f} = cost is {round_trip_cost/m5_median*100:.1f}% of range")

# Sample to verify
print(f"\n=== SAMPLE M5 ROWS (first 5 + last 5) ===")
print(m5[['Open', 'High', 'Low', 'Close', 'TickVolume', 'spread_usd']].head().to_string())
print()
print(m5[['Open', 'High', 'Low', 'Close', 'TickVolume', 'spread_usd']].tail().to_string())

=== M5 TRAINING DATA ===

M1 rows: 1,767,439
M5 rows: 354,152
Period: 2020-01-02 01:00:00 to 2024-12-31 21:55:00

=== CANDLE RANGE COMPARISON (USD) ===

M1 candle range:


KeyError: "['95%'] not in index"

In [7]:
# === RESAMPLE M1 TO M5 (TRAINING DATA ONLY) ===

# Aggregation rules for OHLC + volume + spread
m5 = df.resample('5min').agg({
    'Open':       'first',
    'High':       'max',
    'Low':        'min',
    'Close':      'last',
    'TickVolume': 'sum',
    'Spread':     'mean',  # average spread across the 5 minutes
    'RealVolume': 'sum'
})

# Drop bars with no data (weekends, holidays produce all-NaN rows)
m5 = m5.dropna(subset=['Open', 'High', 'Low', 'Close'])

# Recompute USD spread
m5['spread_usd'] = m5['Spread'] * 0.001

print(f"=== M5 TRAINING DATA ===\n")
print(f"M1 rows: {len(df):,}")
print(f"M5 rows: {len(m5):,}")
print(f"Period: {m5.index.min()} to {m5.index.max()}")

# Compare M1 vs M5 candle stats
print(f"\n=== CANDLE RANGE COMPARISON (USD) ===")
m5['range_usd'] = m5['High'] - m5['Low']
m1_range = (df['High'] - df['Low'])
print(f"\nM1 candle range:")
print(m1_range.describe()[['mean', '50%', '75%', '95%']].round(3).to_string())
print(f"\nM5 candle range:")
print(m5['range_usd'].describe()[['mean', '50%', '75%', '95%']].round(3).to_string())

print(f"\n=== COST-TO-MOVEMENT RATIO ===")
m1_median = m1_range.median()
m5_median = m5['range_usd'].median()
round_trip_cost = 0.66
print(f"M1: median range ${m1_median:.2f}, cost ${round_trip_cost:.2f} = cost is {round_trip_cost/m1_median*100:.1f}% of range")
print(f"M5: median range ${m5_median:.2f}, cost ${round_trip_cost:.2f} = cost is {round_trip_cost/m5_median*100:.1f}% of range")

# Sample to verify
print(f"\n=== SAMPLE M5 ROWS (first 5 + last 5) ===")
print(m5[['Open', 'High', 'Low', 'Close', 'TickVolume', 'spread_usd']].head().to_string())
print()
print(m5[['Open', 'High', 'Low', 'Close', 'TickVolume', 'spread_usd']].tail().to_string())

=== M5 TRAINING DATA ===

M1 rows: 1,767,439
M5 rows: 354,152
Period: 2020-01-02 01:00:00 to 2024-12-31 21:55:00

=== CANDLE RANGE COMPARISON (USD) ===

M1 candle range:


KeyError: "['95%'] not in index"

In [8]:
# === M5 STATS (CORRECTED) ===

m5['range_usd'] = m5['High'] - m5['Low']
m1_range = (df['High'] - df['Low'])

print(f"=== CANDLE RANGE COMPARISON (USD per ounce) ===\n")

print(f"M1 candle range:")
m1_stats = m1_range.describe(percentiles=[0.5, 0.75, 0.95])
print(m1_stats.round(3).to_string())

print(f"\nM5 candle range:")
m5_stats = m5['range_usd'].describe(percentiles=[0.5, 0.75, 0.95])
print(m5_stats.round(3).to_string())

print(f"\n=== COST-TO-MOVEMENT RATIO ===")
m1_median = m1_range.median()
m5_median = m5['range_usd'].median()
round_trip_cost = 0.66

m1_pct = (round_trip_cost / m1_median) * 100
m5_pct = (round_trip_cost / m5_median) * 100
improvement = m1_pct / m5_pct

print(f"M1: median range ${m1_median:.2f}, cost ${round_trip_cost:.2f}")
print(f"    Cost = {m1_pct:.1f}% of typical movement")
print(f"\nM5: median range ${m5_median:.2f}, cost ${round_trip_cost:.2f}")
print(f"    Cost = {m5_pct:.1f}% of typical movement")
print(f"\nM5 cost ratio is {improvement:.1f}x better than M1.")

# Sample to verify data looks right
print(f"\n=== SAMPLE M5 ROWS ===")
print("First 5:")
print(m5[['Open', 'High', 'Low', 'Close', 'TickVolume', 'spread_usd']].head().to_string())
print("\nLast 5:")
print(m5[['Open', 'High', 'Low', 'Close', 'TickVolume', 'spread_usd']].tail().to_string())

=== CANDLE RANGE COMPARISON (USD per ounce) ===

M1 candle range:
count    1767439.000
mean           0.580
std            0.535
min            0.000
50%            0.438
75%            0.722
95%            1.494
max           28.387

M5 candle range:
count    354152.000
mean          1.398
std           1.179
min           0.005
50%           1.087
75%           1.719
95%           3.432
max          31.687

=== COST-TO-MOVEMENT RATIO ===
M1: median range $0.44, cost $0.66
    Cost = 150.7% of typical movement

M5: median range $1.09, cost $0.66
    Cost = 60.7% of typical movement

M5 cost ratio is 2.5x better than M1.

=== SAMPLE M5 ROWS ===
First 5:
                         Open      High       Low     Close  TickVolume  spread_usd
Time                                                                               
2020-01-02 01:00:00  1518.464  1519.624  1517.172  1518.084         366        0.26
2020-01-02 01:05:00  1518.110  1519.163  1517.761  1519.055         242        0.26
20

In [9]:
# === H4 REFORMULATED FOR M5 ===

def calculate_atr_m5(data, period=14):
    high_low = data['High'] - data['Low']
    high_close = abs(data['High'] - data['Close'].shift(1))
    low_close = abs(data['Low'] - data['Close'].shift(1))
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()

def calculate_session_vwap_m5(data, session_start_hour=13):
    data = data.copy()
    data['typical'] = (data['High'] + data['Low'] + data['Close']) / 3
    data['tpv'] = data['typical'] * data['TickVolume']
    data['session_id'] = (
        (data.index.hour == session_start_hour) & 
        (data.index.minute == 0)
    ).cumsum()
    data['cum_tpv'] = data.groupby('session_id')['tpv'].cumsum()
    data['cum_vol'] = data.groupby('session_id')['TickVolume'].cumsum()
    return data['cum_tpv'] / data['cum_vol']

print("Computing indicators on M5 data...")
m5['atr14'] = calculate_atr_m5(m5, 14)
m5['vwap'] = calculate_session_vwap_m5(m5, session_start_hour=13)
m5['dist_from_vwap'] = m5['Close'] - m5['vwap']
m5['hour'] = m5.index.hour
m5['bullish_candle'] = m5['Close'] > m5['Open']
m5['bearish_candle'] = m5['Close'] < m5['Open']
print("Done.\n")

# H4 M5 parameters - properly sized for M5 movement
ATR_MULT_FROM_VWAP = 2.5
ATR_MULT_SL        = 1.5    # wider SL for M5 noise (was 1.0)
ATR_MULT_TP        = 2.5    # bigger TP for proper R:R (was 1.0)
MAX_SPREAD         = 0.40
TIMEOUT_BARS       = 10     # 50 minutes (was 25 M1 bars)
TRADE_HOURS        = [13, 14, 15]
ENTRY_COST_USD     = 0.33
TP_VWAP_FRACTION   = 0.7

trades_m5 = []
i = 50

while i < len(m5) - TIMEOUT_BARS:
    row = m5.iloc[i]
    
    if row['hour'] not in TRADE_HOURS:
        i += 1
        continue
    if row['spread_usd'] > MAX_SPREAD:
        i += 1
        continue
    if pd.isna(row['atr14']) or row['atr14'] < 0.5:
        i += 1
        continue
    if pd.isna(row['vwap']):
        i += 1
        continue
    
    atr = row['atr14']
    dist = row['dist_from_vwap']
    
    direction = None
    if dist > ATR_MULT_FROM_VWAP * atr and row['bearish_candle']:
        direction = 'SELL'
    elif dist < -ATR_MULT_FROM_VWAP * atr and row['bullish_candle']:
        direction = 'BUY'
    
    if direction is None:
        i += 1
        continue
    
    entry_price = row['Close']
    entry_time = m5.index[i]
    target_distance = max(abs(dist) * TP_VWAP_FRACTION, ATR_MULT_TP * atr)
    
    if direction == 'BUY':
        sl = entry_price - ATR_MULT_SL * atr
        tp = entry_price + target_distance
    else:
        sl = entry_price + ATR_MULT_SL * atr
        tp = entry_price - target_distance
    
    exit_price, exit_reason, exit_time = None, None, None
    
    for j in range(i + 1, min(i + 1 + TIMEOUT_BARS, len(m5))):
        bar = m5.iloc[j]
        if direction == 'BUY':
            if bar['Low'] <= sl:
                exit_price, exit_reason, exit_time = sl, 'SL', m5.index[j]
                break
            if bar['High'] >= tp:
                exit_price, exit_reason, exit_time = tp, 'TP', m5.index[j]
                break
        else:
            if bar['High'] >= sl:
                exit_price, exit_reason, exit_time = sl, 'SL', m5.index[j]
                break
            if bar['Low'] <= tp:
                exit_price, exit_reason, exit_time = tp, 'TP', m5.index[j]
                break
    
    if exit_price is None:
        exit_idx = min(i + TIMEOUT_BARS, len(m5) - 1)
        exit_price = m5.iloc[exit_idx]['Close']
        exit_reason = 'TIMEOUT'
        exit_time = m5.index[exit_idx]
    
    if direction == 'BUY':
        gross_pl = exit_price - entry_price
    else:
        gross_pl = entry_price - exit_price
    
    net_pl = gross_pl - (ENTRY_COST_USD * 2)
    
    trades_m5.append({
        'entry_time': entry_time,
        'year': entry_time.year,
        'direction': direction,
        'entry': entry_price,
        'sl': sl,
        'tp': tp,
        'exit_time': exit_time,
        'exit': exit_price,
        'exit_reason': exit_reason,
        'atr': atr,
        'dist_from_vwap': dist,
        'gross_pl_usd': gross_pl,
        'net_pl_usd': net_pl,
        'hour': row['hour']
    })
    
    i = j + 1 if exit_price is not None else i + 1

trades_m5_df = pd.DataFrame(trades_m5)

print(f"=== H4-M5 RESULTS (5 YEARS TRAINING DATA) ===\n")
print(f"Total trades: {len(trades_m5_df):,}\n")

if len(trades_m5_df) > 0:
    def summarize(name, t):
        if len(t) == 0:
            print(f"\n{name}: NO TRADES")
            return
        wins = t[t['net_pl_usd'] > 0]
        losses = t[t['net_pl_usd'] <= 0]
        pf = wins['net_pl_usd'].sum() / abs(losses['net_pl_usd'].sum()) if len(losses) > 0 and losses['net_pl_usd'].sum() != 0 else float('inf')
        print(f"{name}:")
        print(f"  Trades:     {len(t):,}")
        print(f"  Win rate:   {(t['net_pl_usd']>0).mean()*100:.1f}%")
        print(f"  PF:         {pf:.2f}")
        print(f"  Total P/L:  ${t['net_pl_usd'].sum():.2f}")
        print(f"  Per trade:  ${t['net_pl_usd'].mean():.2f}")
        print(f"  Avg win:    ${wins['net_pl_usd'].mean() if len(wins) > 0 else 0:.2f}")
        print(f"  Avg loss:   ${losses['net_pl_usd'].mean() if len(losses) > 0 else 0:.2f}")
        return pf
    
    print("=== OVERALL ===")
    summarize("All trades", trades_m5_df)
    print()
    summarize("BUY only", trades_m5_df[trades_m5_df['direction'] == 'BUY'])
    print()
    summarize("SELL only", trades_m5_df[trades_m5_df['direction'] == 'SELL'])
    
    print("\n\n=== BY YEAR × DIRECTION ===")
    print(f"{'Year':<8}{'Dir':<8}{'Trades':<10}{'Win%':<8}{'PF':<8}{'Total':<10}{'PerTrade':<10}")
    print("-" * 60)
    for year in sorted(trades_m5_df['year'].unique()):
        for direction in ['BUY', 'SELL']:
            subset = trades_m5_df[
                (trades_m5_df['year'] == year) & 
                (trades_m5_df['direction'] == direction)
            ]
            if len(subset) == 0:
                continue
            wins = subset[subset['net_pl_usd'] > 0]
            losses = subset[subset['net_pl_usd'] <= 0]
            pf = wins['net_pl_usd'].sum() / abs(losses['net_pl_usd'].sum()) if len(losses) > 0 and losses['net_pl_usd'].sum() != 0 else float('inf')
            wr = (subset['net_pl_usd'] > 0).mean() * 100
            total = subset['net_pl_usd'].sum()
            per = subset['net_pl_usd'].mean()
            print(f"{year:<8}{direction:<8}{len(subset):<10}{wr:<8.1f}{pf:<8.2f}{total:<10.2f}{per:<10.2f}")
    
    # Trade frequency
    days_traded = trades_m5_df['entry_time'].dt.date.nunique()
    print(f"\nTrading days: {days_traded}")
    print(f"Trades per active day: {len(trades_m5_df)/days_traded:.1f}")
    print(f"\nExit reasons:")
    print(trades_m5_df['exit_reason'].value_counts())

Computing indicators on M5 data...
Done.

=== H4-M5 RESULTS (5 YEARS TRAINING DATA) ===

Total trades: 622

=== OVERALL ===
All trades:
  Trades:     622
  Win rate:   42.3%
  PF:         0.55
  Total P/L:  $-594.07
  Per trade:  $-0.96
  Avg win:    $2.75
  Avg loss:   $-3.67

BUY only:
  Trades:     254
  Win rate:   44.9%
  PF:         0.55
  Total P/L:  $-259.27
  Per trade:  $-1.02
  Avg win:    $2.73
  Avg loss:   $-4.08

SELL only:
  Trades:     368
  Win rate:   40.5%
  PF:         0.55
  Total P/L:  $-334.81
  Per trade:  $-0.91
  Avg win:    $2.76
  Avg loss:   $-3.41


=== BY YEAR × DIRECTION ===
Year    Dir     Trades    Win%    PF      Total     PerTrade  
------------------------------------------------------------
2020    BUY     31        48.4    0.62    -34.45    -1.11     
2020    SELL    67        32.8    0.42    -86.41    -1.29     
2021    BUY     49        38.8    0.42    -64.37    -1.31     
2021    SELL    81        29.6    0.35    -114.62   -1.42     
2022    B

In [10]:
# === HYPOTHESIS 5: OPENING RANGE BREAKOUT (M5, NY Session) ===

# We need M5 data with hour and date columns
m5['date'] = m5.index.date
m5['minute'] = m5.index.minute

# Define session windows
RANGE_START_HOUR = 13      # NY session opens at 13:00 UTC
RANGE_END_HOUR   = 13      # Range window: 13:00-13:30
RANGE_END_MIN    = 30      
SESSION_END_HOUR = 16      # No new trades after 16:00
SESSION_CLOSE    = 16      # Force close at 16:00

MAX_SPREAD     = 0.40
ENTRY_COST_USD = 0.33

trades_orb = []

# Group by date
unique_dates = sorted(m5['date'].unique())
print(f"Processing {len(unique_dates)} trading days...")

for date in unique_dates:
    day_data = m5[m5['date'] == date]
    
    # Find range window: 13:00-13:25 (5 bars covering 13:00, 13:05, 13:10, 13:15, 13:20, 13:25)
    range_window = day_data[
        (day_data.index.hour == RANGE_START_HOUR) & 
        (day_data.index.minute < 30)
    ]
    
    if len(range_window) < 4:  # need at least 4 bars in range window
        continue
    
    range_high = range_window['High'].max()
    range_low  = range_window['Low'].min()
    range_size = range_high - range_low
    
    if range_size < 0.5:  # skip if range too tiny (less than $0.50)
        continue
    
    # Trading window: 13:30 to 16:00 UTC
    trading_window = day_data[
        ((day_data.index.hour == RANGE_START_HOUR) & (day_data.index.minute >= 30)) |
        ((day_data.index.hour > RANGE_START_HOUR) & (day_data.index.hour < SESSION_CLOSE))
    ]
    
    if len(trading_window) == 0:
        continue
    
    # Look for first breakout
    direction = None
    entry_price = None
    entry_time = None
    
    for idx, bar in trading_window.iterrows():
        if bar['spread_usd'] > MAX_SPREAD:
            continue
        
        # Closing-basis break: bar must close beyond range
        if bar['Close'] > range_high and direction is None:
            direction = 'BUY'
            entry_price = bar['Close']
            entry_time = idx
            break
        elif bar['Close'] < range_low and direction is None:
            direction = 'SELL'
            entry_price = bar['Close']
            entry_time = idx
            break
    
    if direction is None:
        continue  # no breakout that day
    
    # Set SL and TP
    if direction == 'BUY':
        sl = range_low
        tp = entry_price + range_size
    else:
        sl = range_high
        tp = entry_price - range_size
    
    # Find exit by walking forward through remaining bars
    remaining = day_data[day_data.index > entry_time]
    remaining = remaining[remaining.index.hour < SESSION_CLOSE]
    
    exit_price, exit_reason, exit_time = None, None, None
    
    for idx, bar in remaining.iterrows():
        if direction == 'BUY':
            if bar['Low'] <= sl:
                exit_price, exit_reason, exit_time = sl, 'SL', idx
                break
            if bar['High'] >= tp:
                exit_price, exit_reason, exit_time = tp, 'TP', idx
                break
        else:
            if bar['High'] >= sl:
                exit_price, exit_reason, exit_time = sl, 'SL', idx
                break
            if bar['Low'] <= tp:
                exit_price, exit_reason, exit_time = tp, 'TP', idx
                break
    
    # If no SL or TP hit, close at session end
    if exit_price is None:
        if len(remaining) > 0:
            last_bar = remaining.iloc[-1]
            exit_price = last_bar['Close']
            exit_reason = 'SESSION_END'
            exit_time = remaining.index[-1]
        else:
            continue  # shouldn't happen but skip if it does
    
    if direction == 'BUY':
        gross_pl = exit_price - entry_price
    else:
        gross_pl = entry_price - exit_price
    
    net_pl = gross_pl - (ENTRY_COST_USD * 2)
    
    trades_orb.append({
        'date': date,
        'entry_time': entry_time,
        'year': entry_time.year,
        'direction': direction,
        'range_high': range_high,
        'range_low': range_low,
        'range_size': range_size,
        'entry': entry_price,
        'sl': sl,
        'tp': tp,
        'exit_time': exit_time,
        'exit': exit_price,
        'exit_reason': exit_reason,
        'gross_pl_usd': gross_pl,
        'net_pl_usd': net_pl
    })

trades_orb_df = pd.DataFrame(trades_orb)

print(f"\n=== ORB M5 RESULTS (5 YEARS TRAINING DATA) ===\n")
print(f"Total trades: {len(trades_orb_df):,}\n")

if len(trades_orb_df) > 0:
    def summarize(name, t):
        if len(t) == 0:
            print(f"\n{name}: NO TRADES")
            return
        wins = t[t['net_pl_usd'] > 0]
        losses = t[t['net_pl_usd'] <= 0]
        pf = wins['net_pl_usd'].sum() / abs(losses['net_pl_usd'].sum()) if len(losses) > 0 and losses['net_pl_usd'].sum() != 0 else float('inf')
        print(f"{name}:")
        print(f"  Trades:     {len(t):,}")
        print(f"  Win rate:   {(t['net_pl_usd']>0).mean()*100:.1f}%")
        print(f"  PF:         {pf:.2f}")
        print(f"  Total P/L:  ${t['net_pl_usd'].sum():.2f}")
        print(f"  Per trade:  ${t['net_pl_usd'].mean():.2f}")
        print(f"  Avg win:    ${wins['net_pl_usd'].mean() if len(wins) > 0 else 0:.2f}")
        print(f"  Avg loss:   ${losses['net_pl_usd'].mean() if len(losses) > 0 else 0:.2f}")
        return pf
    
    print("=== OVERALL ===")
    summarize("All trades", trades_orb_df)
    print()
    summarize("BUY only", trades_orb_df[trades_orb_df['direction'] == 'BUY'])
    print()
    summarize("SELL only", trades_orb_df[trades_orb_df['direction'] == 'SELL'])
    
    print("\n\n=== BY YEAR × DIRECTION ===")
    print(f"{'Year':<8}{'Dir':<8}{'Trades':<10}{'Win%':<8}{'PF':<8}{'Total':<10}{'PerTrade':<10}")
    print("-" * 60)
    for year in sorted(trades_orb_df['year'].unique()):
        for direction in ['BUY', 'SELL']:
            subset = trades_orb_df[
                (trades_orb_df['year'] == year) & 
                (trades_orb_df['direction'] == direction)
            ]
            if len(subset) == 0:
                continue
            wins = subset[subset['net_pl_usd'] > 0]
            losses = subset[subset['net_pl_usd'] <= 0]
            pf = wins['net_pl_usd'].sum() / abs(losses['net_pl_usd'].sum()) if len(losses) > 0 and losses['net_pl_usd'].sum() != 0 else float('inf')
            wr = (subset['net_pl_usd'] > 0).mean() * 100
            total = subset['net_pl_usd'].sum()
            per = subset['net_pl_usd'].mean()
            print(f"{year:<8}{direction:<8}{len(subset):<10}{wr:<8.1f}{pf:<8.2f}{total:<10.2f}{per:<10.2f}")
    
    print(f"\nExit reasons:")
    print(trades_orb_df['exit_reason'].value_counts())
    
    print(f"\nRange size statistics:")
    print(trades_orb_df['range_size'].describe(percentiles=[0.5, 0.75, 0.95]).round(2).to_string())

Processing 1552 trading days...

=== ORB M5 RESULTS (5 YEARS TRAINING DATA) ===

Total trades: 1,241

=== OVERALL ===
All trades:
  Trades:     1,241
  Win rate:   52.4%
  PF:         0.84
  Total P/L:  $-522.66
  Per trade:  $-0.42
  Avg win:    $4.21
  Avg loss:   $-5.52

BUY only:
  Trades:     639
  Win rate:   52.3%
  PF:         0.83
  Total P/L:  $-277.57
  Per trade:  $-0.43
  Avg win:    $4.02
  Avg loss:   $-5.31

SELL only:
  Trades:     602
  Win rate:   52.5%
  PF:         0.85
  Total P/L:  $-245.09
  Per trade:  $-0.41
  Avg win:    $4.42
  Avg loss:   $-5.74


=== BY YEAR × DIRECTION ===
Year    Dir     Trades    Win%    PF      Total     PerTrade  
------------------------------------------------------------
2020    BUY     119       52.9    0.97    -7.99     -0.07     
2020    SELL    103       52.4    1.03    8.32      0.08      
2021    BUY     131       48.1    0.64    -114.60   -0.87     
2021    SELL    120       49.2    0.76    -77.32    -0.64     
2022    BUY  

In [11]:
# === ORB DIAGNOSTICS ===

print("=== Performance by Range Size Bucket ===\n")

trades_orb_df['range_bucket'] = pd.cut(
    trades_orb_df['range_size'],
    bins=[0, 3, 5, 8, 12, 100],
    labels=['Tiny (<$3)', 'Small ($3-5)', 'Med ($5-8)', 'Large ($8-12)', 'Huge ($12+)']
)

by_range = trades_orb_df.groupby('range_bucket', observed=True).agg(
    trades=('net_pl_usd', 'count'),
    win_rate=('net_pl_usd', lambda x: (x > 0).mean() * 100),
    avg_pl=('net_pl_usd', 'mean'),
    total_pl=('net_pl_usd', 'sum')
)
print(by_range.to_string())

print("\n\n=== Performance by Exit Reason ===\n")
exit_perf = trades_orb_df.groupby('exit_reason').agg(
    trades=('net_pl_usd', 'count'),
    avg_pl=('net_pl_usd', 'mean'),
    total_pl=('net_pl_usd', 'sum')
)
print(exit_perf.to_string())

print("\n\n=== SESSION_END trades — are they killing us? ===")
session_end = trades_orb_df[trades_orb_df['exit_reason'] == 'SESSION_END']
print(f"Count: {len(session_end)}")
print(f"Win rate at session end: {(session_end['net_pl_usd'] > 0).mean()*100:.1f}%")
print(f"Total P/L: ${session_end['net_pl_usd'].sum():.2f}")
print(f"Avg P/L:   ${session_end['net_pl_usd'].mean():.2f}")

print("\n\n=== If we DROP session_end exits (TP or SL only) ===")
no_session_end = trades_orb_df[trades_orb_df['exit_reason'] != 'SESSION_END']
if len(no_session_end) > 0:
    wins = no_session_end[no_session_end['net_pl_usd'] > 0]
    losses = no_session_end[no_session_end['net_pl_usd'] <= 0]
    pf = wins['net_pl_usd'].sum() / abs(losses['net_pl_usd'].sum()) if len(losses) > 0 else float('inf')
    print(f"Trades: {len(no_session_end)}")
    print(f"Win rate: {(no_session_end['net_pl_usd']>0).mean()*100:.1f}%")
    print(f"PF: {pf:.2f}")
    print(f"Total: ${no_session_end['net_pl_usd'].sum():.2f}")
    print(f"Per trade: ${no_session_end['net_pl_usd'].mean():.2f}")

print("\n\n=== Performance by hour of breakout ===")
trades_orb_df['entry_hour'] = pd.to_datetime(trades_orb_df['entry_time']).dt.hour
by_hour = trades_orb_df.groupby('entry_hour').agg(
    trades=('net_pl_usd', 'count'),
    win_rate=('net_pl_usd', lambda x: (x > 0).mean() * 100),
    avg_pl=('net_pl_usd', 'mean'),
    total_pl=('net_pl_usd', 'sum')
)
print(by_hour.to_string())

=== Performance by Range Size Bucket ===

               trades   win_rate    avg_pl  total_pl
range_bucket                                        
Tiny (<$3)        159  55.345912 -0.550610   -87.547
Small ($3-5)      444  56.081081 -0.543367  -241.255
Med ($5-8)        424  48.113208 -0.577939  -245.046
Large ($8-12)     166  50.602410  0.178590    29.646
Huge ($12+)        48  52.083333  0.448792    21.542


=== Performance by Exit Reason ===

             trades    avg_pl  total_pl
exit_reason                            
SESSION_END     314 -1.531369  -480.850
SL              375 -6.783752 -2543.907
TP              552  4.532784  2502.097


=== SESSION_END trades — are they killing us? ===
Count: 314
Win rate at session end: 31.2%
Total P/L: $-480.85
Avg P/L:   $-1.53


=== If we DROP session_end exits (TP or SL only) ===
Trades: 927
Win rate: 59.5%
PF: 0.98
Total: $-41.81
Per trade: $-0.05


=== Performance by hour of breakout ===
            trades   win_rate    avg_pl  total_pl
